# 07 - Entrenamiento y seleccion del modelo final

Este notebook reconstruye la fase de entrenamiento del TFG. Compara modelos, selecciona el ganador y calibra la politica final usando solo train, validacion cruzada agrupada y predicciones OOF. El test temporal queda reservado para `08_model_evaluation.ipynb`.

## 1. Setup y rutas

Las rutas se importan desde `triaje_ia.config` para mantener consistencia con el proyecto. No se crean modulos nuevos en `src/`; toda la reconstruccion metodologica queda en este notebook.

In [2]:
# ruff: noqa: E402, I001
import hashlib
import json
import sys
import time
import warnings
from collections.abc import Callable
from pathlib import Path
from typing import Any

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from scipy.optimize import minimize
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_recall_fscore_support
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("No se pudo localizar la raiz del proyecto")


PROJECT_DISCOVERED = find_project_root(Path.cwd().resolve())
SRC_DIR = PROJECT_DISCOVERED / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from triaje_ia.config import DATA_INTERIM, DATA_PROCESSED, MODELS_DIR, PROJECT_ROOT  # noqa: E402

assert PROJECT_ROOT == PROJECT_DISCOVERED
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
CV_N_SPLITS = 5
THRESHOLD_A1 = 0.20
assert THRESHOLD_A1 >= 0.20
np.random.seed(RANDOM_STATE)

BASE = {
    "X_train": DATA_PROCESSED / "X_train.parquet",
    "X_test": DATA_PROCESSED / "X_test.parquet",
    "y_train": DATA_PROCESSED / "y_train.parquet",
    "groups_train": DATA_PROCESSED / "groups_train.npy",
    "sample_weights_train": DATA_PROCESSED / "sample_weights_train.npy",
    "feature_config_selected": DATA_PROCESSED / "feature_config_selected.json",
    "feature_config_exp5b": DATA_PROCESSED / "feature_config_exp5b.json",
    "dataset_clean": DATA_INTERIM / "dataset_clean.parquet",
    "dataset_features": DATA_PROCESSED / "dataset_features.parquet",
}
NLP = {
    "llm_train": DATA_PROCESSED / "llm_features_train.parquet",
    "llm_test": DATA_PROCESSED / "llm_features_test.parquet",
    "bert_train": DATA_PROCESSED / "bert_embeddings_train.parquet",
    "bert_test": DATA_PROCESSED / "bert_embeddings_test.parquet",
    "bert_svd": DATA_PROCESSED / "bert_svd.joblib",
    "bert_svd_legacy": DATA_PROCESSED / "svd_bert.joblib",
}
OUT = {
    "model": MODELS_DIR / "lgbm_bert_final.joblib",
    "thresholds": MODELS_DIR / "thresholds.json",
    "feature_list": MODELS_DIR / "feature_list.json",
    "cv_results": DATA_PROCESSED / "cv_results.parquet",
    "oof_predictions": DATA_PROCESSED / "oof_predictions.parquet",
    "metadata": MODELS_DIR / "model_training_metadata.json",
}
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Proyecto: {PROJECT_ROOT}")
for name, path in OUT.items():
    print(f"{name:<16} -> {path.relative_to(PROJECT_ROOT)}")

Proyecto: C:\Users\CARLOS\triaje-ia-tfg
model            -> models\lgbm_bert_final.joblib
thresholds       -> models\thresholds.json
feature_list     -> models\feature_list.json
cv_results       -> data\processed\cv_results.parquet
oof_predictions  -> data\processed\oof_predictions.parquet
metadata         -> models\model_training_metadata.json


## 2. Carga de datos y validaciones

Se cargan `X_train`, `y_train`, `groups_train` y `sample_weights_train`. `X_test` solo se carga para comprobar columnas y shapes; no se carga `y_test` ni se calcula ninguna metrica de test.

In [3]:
missing = [
    p
    for p in [
        BASE["X_train"],
        BASE["X_test"],
        BASE["y_train"],
        BASE["groups_train"],
        BASE["sample_weights_train"],
        BASE["feature_config_selected"],
    ]
    if not p.exists()
]
if missing:
    raise FileNotFoundError(
        "Faltan artefactos base: " + ", ".join(str(p) for p in missing)
    )

X_train = pd.read_parquet(BASE["X_train"])
X_test = pd.read_parquet(BASE["X_test"])
y_train = pd.read_parquet(BASE["y_train"]).squeeze().astype(int).reset_index(drop=True)
groups_train = np.load(BASE["groups_train"])
sample_weights_train = np.load(BASE["sample_weights_train"])
config_selected = json.loads(
    BASE["feature_config_selected"].read_text(encoding="utf-8")
)
FEATURES_BASE = config_selected["features_seleccionadas"]

PROHIBIDAS = {
    "acuity",
    "diagnosis",
    "icd_code",
    "icd_title",
    "ccs",
    "ccsr",
    "disposition",
    "outcome",
    "hospital_expire_flag",
    "subject_id",
    "stay_id",
}
assert len(X_train) == len(y_train) == len(groups_train) == len(sample_weights_train)
assert y_train.isin([1, 2, 3, 4, 5]).all()
assert not X_train.columns.duplicated().any() and not X_test.columns.duplicated().any()
assert set(FEATURES_BASE).issubset(X_train.columns) and set(FEATURES_BASE).issubset(
    X_test.columns
)
assert PROHIBIDAS.isdisjoint(FEATURES_BASE)
assert (sample_weights_train > 0).all() and not np.isnan(sample_weights_train).any()

X_base_train = X_train[FEATURES_BASE].reset_index(drop=True)
X_base_test = X_test[FEATURES_BASE].reset_index(drop=True)
print(
    f"Base train: {X_base_train.shape} | "
    f"Base test: {X_base_test.shape} (solo shape/columnas)"
)
print(y_train.value_counts(normalize=True).sort_index().round(4).to_string())

Base train: (334480, 46) | Base test: (83620, 46) (solo shape/columnas)
acuity
1    0.0579
2    0.3326
3    0.5372
4    0.0695
5    0.0028


## 3. Protocolo experimental

La metrica principal es Macro F1. Como metricas clinicas secundarias se guardan precision y recall de Acuity 1 y 2. La validacion usa `StratifiedGroupKFold` con `groups_train`, de forma que un mismo paciente no aparece en train y validacion dentro de un fold.

In [4]:
CLASSES = np.array([1, 2, 3, 4, 5])
cv = StratifiedGroupKFold(n_splits=CV_N_SPLITS, shuffle=False)


def macro_f1(y_true, y_pred) -> float:
    return float(f1_score(y_true, y_pred, average="macro", zero_division=0))


def metricas(y_true, y_pred) -> dict[str, float]:
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=CLASSES, zero_division=0
    )
    return {
        "macro_f1": macro_f1(y_true, y_pred),
        "recall_a1": float(r[0]),
        "precision_a1": float(p[0]),
        "recall_a2": float(r[1]),
        "precision_a2": float(p[1]),
    }


def align_proba(proba: np.ndarray, classes: np.ndarray) -> np.ndarray:
    out = np.zeros((len(proba), 5), dtype=float)
    for j, cls in enumerate(classes):
        out[:, int(cls) - 1] = proba[:, j]
    sums = np.where(
        out.sum(axis=1, keepdims=True) == 0, 1, out.sum(axis=1, keepdims=True)
    )
    return out / sums


for fold, (_, idx_val) in enumerate(
    cv.split(X_base_train, y_train, groups=groups_train), start=1
):
    n_patients = len(np.unique(groups_train[idx_val]))
    dist_fold = y_train.iloc[idx_val].value_counts(normalize=True)
    dist_fold = dist_fold.sort_index().round(3).to_dict()
    print(f"Fold {fold}: n={len(idx_val):,}, pacientes={n_patients:,}")
    print(f"  dist={dist_fold}")

Fold 1: n=66,897, pacientes=33,737
  dist={1: 0.058, 2: 0.333, 3: 0.537, 4: 0.069, 5: 0.003}
Fold 2: n=66,896, pacientes=33,711
  dist={1: 0.058, 2: 0.333, 3: 0.537, 4: 0.069, 5: 0.003}
Fold 3: n=66,897, pacientes=33,713
  dist={1: 0.058, 2: 0.333, 3: 0.537, 4: 0.069, 5: 0.003}
Fold 4: n=66,895, pacientes=33,725
  dist={1: 0.058, 2: 0.333, 3: 0.537, 4: 0.069, 5: 0.003}
Fold 5: n=66,895, pacientes=33,710
  dist={1: 0.058, 2: 0.333, 3: 0.537, 4: 0.069, 5: 0.003}


## 4. Helpers OOF

Estas funciones entrenan cada candidato por folds y devuelven predicciones OOF. Todo preprocesador se ajusta dentro del fold con train y solo transforma validacion.

In [5]:
def cargar_lgbm_params() -> dict[str, Any]:
    for path in [
        MODELS_DIR / "lgbm_best_params.json",
        MODELS_DIR / "mejores_parametros_lightgbm.json",
    ]:
        if path.exists():
            params = json.loads(path.read_text(encoding="utf-8"))
            return {k: v for k, v in params.items() if not k.startswith("_")}
    return {
        "objective": "multiclass",
        "num_class": 5,
        "n_estimators": 1000,
        "learning_rate": 0.03,
        "num_leaves": 63,
        "min_child_samples": 50,
        "subsample": 0.8,
        "subsample_freq": 1,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
    }


LGBM_PARAMS = cargar_lgbm_params()
LGBM_PARAMS.update(
    {
        "objective": "multiclass",
        "num_class": 5,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "verbose": -1,
    }
)


def evaluar_sklearn(
    nombre: str,
    factory: Callable[[], Any],
    x_data: pd.DataFrame,
    sample_weight_param: str | None = None,
):
    assert (
        len(x_data) == len(y_train)
        and not x_data.columns.duplicated().any()
        and PROHIBIDAS.isdisjoint(x_data.columns)
    )
    pred = np.zeros(len(y_train), dtype=int)
    proba = np.zeros((len(y_train), 5), dtype=float)
    folds = []
    for fold, (idx_tr, idx_val) in enumerate(
        cv.split(x_data, y_train, groups=groups_train), start=1
    ):
        model = factory()
        kwargs = (
            {sample_weight_param: sample_weights_train[idx_tr]}
            if sample_weight_param
            else {}
        )
        model.fit(x_data.iloc[idx_tr], y_train.iloc[idx_tr], **kwargs)
        pred[idx_val] = model.predict(x_data.iloc[idx_val]).astype(int)
        proba[idx_val] = align_proba(
            model.predict_proba(x_data.iloc[idx_val]), np.asarray(model.classes_)
        )
        folds.append(
            {
                "modelo": nombre,
                "fold": fold,
                **metricas(y_train.iloc[idx_val], pred[idx_val]),
            }
        )
        print(f"{nombre:<26} fold {fold}: {folds[-1]['macro_f1']:.4f}")
    return (
        {"modelo": nombre, "n_features": x_data.shape[1], **metricas(y_train, pred)},
        pd.DataFrame(
            {
                "modelo": nombre,
                "y_true": y_train,
                "y_pred": pred,
                **{f"proba_{i + 1}": proba[:, i] for i in range(5)},
            }
        ),
        proba,
        folds,
    )


def evaluar_lgbm(nombre: str, x_data: pd.DataFrame):
    assert (
        len(x_data) == len(y_train)
        and not x_data.columns.duplicated().any()
        and PROHIBIDAS.isdisjoint(x_data.columns)
    )
    pred = np.zeros(len(y_train), dtype=int)
    proba = np.zeros((len(y_train), 5), dtype=float)
    best_iters, folds = [], []
    for fold, (idx_tr, idx_val) in enumerate(
        cv.split(x_data, y_train, groups=groups_train), start=1
    ):
        model = LGBMClassifier(**LGBM_PARAMS)
        model.fit(
            x_data.iloc[idx_tr],
            y_train.iloc[idx_tr] - 1,
            sample_weight=sample_weights_train[idx_tr],
            eval_set=[(x_data.iloc[idx_val], y_train.iloc[idx_val] - 1)],
            eval_sample_weight=[sample_weights_train[idx_val]],
            callbacks=[
                lgb.early_stopping(50, verbose=False),
                lgb.log_evaluation(period=-1),
            ],
        )
        proba[idx_val] = model.predict_proba(x_data.iloc[idx_val])
        pred[idx_val] = np.argmax(proba[idx_val], axis=1) + 1
        best_iters.append(
            int(model.best_iteration_ or LGBM_PARAMS.get("n_estimators", 1000))
        )
        folds.append(
            {
                "modelo": nombre,
                "fold": fold,
                "best_iteration": best_iters[-1],
                **metricas(y_train.iloc[idx_val], pred[idx_val]),
            }
        )
        print(f"{nombre:<26} fold {fold}: {folds[-1]['macro_f1']:.4f}")
    return (
        {
            "modelo": nombre,
            "n_features": x_data.shape[1],
            "best_iteration_mean": float(np.mean(best_iters)),
            **metricas(y_train, pred),
        },
        pd.DataFrame(
            {
                "modelo": nombre,
                "y_true": y_train,
                "y_pred": pred,
                **{f"proba_{i + 1}": proba[:, i] for i in range(5)},
            }
        ),
        proba,
        folds,
    )

## 5. Baselines tabulares

Se evalua DummyClassifier, Regresion Logistica, Random Forest y LightGBM con las 46 variables base. Esta comparativa fija el punto de partida antes de incorporar variables rescatadas y NLP.

In [6]:
def columnas_cont_bin(x_data: pd.DataFrame):
    cont = [c for c in x_data.columns if x_data[c].nunique(dropna=True) > 2]
    bin_ = [c for c in x_data.columns if c not in cont]
    return cont, bin_


def make_lr(x_data: pd.DataFrame):
    cont, bin_ = columnas_cont_bin(x_data)
    prep = ColumnTransformer(
        [
            (
                "cont",
                Pipeline(
                    [
                        ("imp", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                cont,
            ),
            ("bin", SimpleImputer(strategy="most_frequent"), bin_),
        ]
    )
    return lambda: Pipeline(
        [
            ("prep", clone(prep)),
            (
                "modelo",
                LogisticRegression(
                    solver="saga",
                    max_iter=1000,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    )


def make_rf(x_data: pd.DataFrame):
    cont, bin_ = columnas_cont_bin(x_data)
    prep = ColumnTransformer(
        [
            ("cont", SimpleImputer(strategy="median"), cont),
            ("bin", SimpleImputer(strategy="most_frequent"), bin_),
        ]
    )
    return lambda: Pipeline(
        [
            ("prep", clone(prep)),
            (
                "modelo",
                RandomForestClassifier(
                    n_estimators=300, max_depth=20, random_state=RANDOM_STATE, n_jobs=-1
                ),
            ),
        ]
    )


def validar_step_modelo(factory: Callable[[], Pipeline]) -> Callable[[], Pipeline]:
    pipeline = factory()
    if "modelo" not in pipeline.named_steps:
        raise ValueError(
            "RandomForest requiere un Pipeline con step 'modelo' "
            "para pasar modelo__sample_weight."
        )
    return factory


resultados, oof_tables, oof_probas = [], [], {}
for overall, oof, proba, _ in [
    evaluar_sklearn(
        "dummy_stratified",
        lambda: DummyClassifier(strategy="stratified", random_state=RANDOM_STATE),
        X_base_train,
    ),
    evaluar_sklearn("logistic_base", make_lr(X_base_train), X_base_train),
    evaluar_sklearn(
        "random_forest_base",
        validar_step_modelo(make_rf(X_base_train)),
        X_base_train,
        "modelo__sample_weight",
    ),
    evaluar_lgbm("lightgbm_base_46", X_base_train),
]:
    resultados.append(overall)
    oof_tables.append(oof)
    oof_probas[overall["modelo"]] = proba

pd.DataFrame(resultados).sort_values("macro_f1", ascending=False)

dummy_stratified           fold 1: 0.2016
dummy_stratified           fold 2: 0.1966
dummy_stratified           fold 3: 0.2010
dummy_stratified           fold 4: 0.1990
dummy_stratified           fold 5: 0.2027
logistic_base              fold 1: 0.3736
logistic_base              fold 2: 0.3701
logistic_base              fold 3: 0.3751
logistic_base              fold 4: 0.3699
logistic_base              fold 5: 0.3730
random_forest_base         fold 1: 0.4619
random_forest_base         fold 2: 0.4673
random_forest_base         fold 3: 0.4708
random_forest_base         fold 4: 0.4685
random_forest_base         fold 5: 0.4611
lightgbm_base_46           fold 1: 0.4876
lightgbm_base_46           fold 2: 0.4792
lightgbm_base_46           fold 3: 0.4944
lightgbm_base_46           fold 4: 0.4860
lightgbm_base_46           fold 5: 0.4885


,modelo,n_features,macro_f1,recall_a1,precision_a1,recall_a2,precision_a2,best_iteration_mean
3,lightgbm_base_46,46,0.487213,0.652367,0.666192,0.602506,0.625558,402.6
2,random_forest_base,46,0.465950,0.566982,0.732005,0.595423,0.622895,NaN
1,logistic_base,46,0.372351,0.714418,0.341527,0.540184,0.588387,NaN
0,dummy_stratified,46,0.200180,0.057715,0.058951,0.336260,0.333782,NaN


## 6. Ablacion tabular: 46 variables frente a exp5b

Se carga exp5b si existe. Si no existe, se reconstruye desde `dataset_features.parquet` y `dataset_clean.parquet`, manteniendo la decision dentro de train. Las variables prohibidas siguen excluidas.

In [7]:
EXP5B_DEFAULT = [
    "n_visitas_previas",
    "visitas_ultimo_mes",
    "visitas_ultimo_a?o",
    "dias_desde_ultima_visita",
    "primera_visita",
    "frecuentador",
    "cc_cefalea",
    "cc_intoxicacion",
    "cc_infeccioso",
    "cc_hemorragia_activa",
    "hx_respiratorio",
    "hx_neuro",
    "hx_psiquiatrico",
    "hx_abuso_sustancias",
    "hx_digestivo",
    "hx_metabolico_renal",
    "hx_infeccioso",
    "hx_trauma_muscular",
]


def reconstruir_exp5b():
    train_p, test_p = (
        DATA_PROCESSED / "X_train_exp5b.parquet",
        DATA_PROCESSED / "X_test_exp5b.parquet",
    )
    if train_p.exists() and test_p.exists() and BASE["feature_config_exp5b"].exists():
        cfg = json.loads(BASE["feature_config_exp5b"].read_text(encoding="utf-8"))
        return (
            pd.read_parquet(train_p).reset_index(drop=True),
            pd.read_parquet(test_p).reset_index(drop=True),
            cfg,
        )
    if not BASE["dataset_features"].exists() or not BASE["dataset_clean"].exists():
        raise FileNotFoundError("Faltan prerequisitos para reconstruir exp5b")
    df_feat = pd.read_parquet(BASE["dataset_features"])
    df_clean = pd.read_parquet(BASE["dataset_clean"])
    df_clean["intime"] = pd.to_datetime(df_clean["intime"], errors="raise")
    cut = df_clean["intime"].quantile(0.80)
    tr_full = df_feat.loc[(df_clean["intime"] <= cut).values].reset_index(drop=True)
    te_full = df_feat.loc[(df_clean["intime"] > cut).values].reset_index(drop=True)
    assert len(tr_full) == len(X_train) and len(te_full) == len(X_test)
    exp4 = [
        c
        for c in ["temperature_valor", "heartrate_valor", "dbp_valor"]
        if c in tr_full.columns
    ]
    extra = [c for c in EXP5B_DEFAULT if c in tr_full.columns]
    xtr = pd.concat(
        [X_base_train, tr_full[exp4 + extra].fillna(0).reset_index(drop=True)], axis=1
    )
    xte = pd.concat(
        [X_base_test, te_full[exp4 + extra].fillna(0).reset_index(drop=True)], axis=1
    )
    cfg = {
        "todas_features": list(xtr.columns),
        "features_nuevas": exp4 + extra,
        "reconstruido_en_notebook_07": True,
    }
    xtr.to_parquet(train_p, index=False)
    xte.to_parquet(test_p, index=False)
    BASE["feature_config_exp5b"].write_text(
        json.dumps(cfg, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    return xtr, xte, cfg


X_exp5b_train, X_exp5b_test, cfg_exp5b = reconstruir_exp5b()
assert len(X_exp5b_train) == len(y_train) and len(X_exp5b_test) == len(X_test)
assert list(X_exp5b_train.columns) == list(X_exp5b_test.columns)
assert not X_exp5b_train.columns.duplicated().any() and PROHIBIDAS.isdisjoint(
    X_exp5b_train.columns
)

overall, oof, proba, _ = evaluar_lgbm("lightgbm_exp5b", X_exp5b_train)
resultados.append(overall)
oof_tables.append(oof)
oof_probas[overall["modelo"]] = proba
print("Variables exp5b a?adidas:", cfg_exp5b.get("features_nuevas", []))

lightgbm_exp5b             fold 1: 0.5187
lightgbm_exp5b             fold 2: 0.5116
lightgbm_exp5b             fold 3: 0.5255
lightgbm_exp5b             fold 4: 0.5211
lightgbm_exp5b             fold 5: 0.5222
Variables exp5b a?adidas: ['n_visitas_previas', 'visitas_ultimo_mes', 'visitas_ultimo_año', 'dias_desde_ultima_visita', 'primera_visita', 'frecuentador', 'cc_cefalea', 'cc_intoxicacion', 'cc_infeccioso', 'cc_hemorragia_activa', 'hx_respiratorio', 'hx_neuro', 'hx_psiquiatrico', 'hx_abuso_sustancias', 'hx_digestivo', 'hx_metabolico_renal', 'hx_infeccioso', 'hx_trauma_muscular']


## 7. Competicion NLP: LLM frente a BERT/SVD

Se cargan las salidas del notebook 06. Se compara LightGBM + exp5b + LLM frente a LightGBM + exp5b + BERT/SVD. La seleccion se hace por OOF; test solo queda preparado para validar columnas.

In [8]:
missing_nlp = [
    p
    for p in [NLP["llm_train"], NLP["llm_test"], NLP["bert_train"], NLP["bert_test"]]
    if not p.exists()
]
if missing_nlp:
    raise FileNotFoundError(
        "Faltan artefactos del notebook 06: "
        + ", ".join(str(p.relative_to(PROJECT_ROOT)) for p in missing_nlp)
    )

llm_train = pd.read_parquet(NLP["llm_train"]).reset_index(drop=True)
llm_test = pd.read_parquet(NLP["llm_test"]).reset_index(drop=True)
bert_train = pd.read_parquet(NLP["bert_train"]).reset_index(drop=True)
bert_test = pd.read_parquet(NLP["bert_test"]).reset_index(drop=True)
bert_svd_path = (
    NLP["bert_svd"]
    if NLP["bert_svd"].exists()
    else (NLP["bert_svd_legacy"] if NLP["bert_svd_legacy"].exists() else None)
)

for name, tr, te in [("llm", llm_train, llm_test), ("bert", bert_train, bert_test)]:
    assert len(tr) == len(X_exp5b_train) and len(te) == len(X_exp5b_test), name
    assert list(tr.columns) == list(te.columns), name
    assert not tr.columns.duplicated().any() and not tr.isna().any().any(), name
    assert PROHIBIDAS.isdisjoint(tr.columns), name

X_llm_train = pd.concat([X_exp5b_train.reset_index(drop=True), llm_train], axis=1)
X_llm_test = pd.concat([X_exp5b_test.reset_index(drop=True), llm_test], axis=1)
X_bert_train = pd.concat([X_exp5b_train.reset_index(drop=True), bert_train], axis=1)
X_bert_test = pd.concat([X_exp5b_test.reset_index(drop=True), bert_test], axis=1)
assert list(X_llm_train.columns) == list(X_llm_test.columns)
assert list(X_bert_train.columns) == list(X_bert_test.columns)

for name, Xcand in [
    ("lightgbm_exp5b_llm", X_llm_train),
    ("lightgbm_exp5b_bert", X_bert_train),
]:
    overall, oof, proba, _ = evaluar_lgbm(name, Xcand)
    resultados.append(overall)
    oof_tables.append(oof)
    oof_probas[overall["modelo"]] = proba

cv_results = (
    pd.DataFrame(resultados)
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)
cv_results

lightgbm_exp5b_llm         fold 1: 0.5476
lightgbm_exp5b_llm         fold 2: 0.5500
lightgbm_exp5b_llm         fold 3: 0.5470
lightgbm_exp5b_llm         fold 4: 0.5450
lightgbm_exp5b_llm         fold 5: 0.5589
lightgbm_exp5b_bert        fold 1: 0.5770
lightgbm_exp5b_bert        fold 2: 0.5747
lightgbm_exp5b_bert        fold 3: 0.5770
lightgbm_exp5b_bert        fold 4: 0.5687
lightgbm_exp5b_bert        fold 5: 0.5798


,modelo,n_features,macro_f1,recall_a1,precision_a1,recall_a2,precision_a2,best_iteration_mean
0,lightgbm_exp5b_bert,82,0.575340,0.705126,0.675052,0.659084,0.670486,470.0
1,lightgbm_exp5b_llm,91,0.549744,0.674152,0.693153,0.643832,0.648929,467.8
2,lightgbm_exp5b,67,0.519848,0.668732,0.696040,0.628921,0.635842,473.6
3,lightgbm_base_46,46,0.487213,0.652367,0.666192,0.602506,0.625558,402.6
4,random_forest_base,46,0.465950,0.566982,0.732005,0.595423,0.622895,NaN
5,logistic_base,46,0.372351,0.714418,0.341527,0.540184,0.588387,NaN
6,dummy_stratified,46,0.200180,0.057715,0.058951,0.336260,0.333782,NaN


## 8. Seleccion del ganador y calibracion OOF

El ganador se escoge por Macro F1 OOF. Despues se ajustan pesos de clase con Nelder-Mead usando solo probabilidades OOF. La regla A1 se aplica antes de los pesos: si `P(A1) >= 0.20`, la prediccion final es Acuity 1.

El valor `THRESHOLD_A1 = 0.20` se mantiene como politica clinica conservadora heredada de los experimentos previos y del dise?o de seguridad del sistema. No se optimiza sobre test en este notebook.

In [9]:
MATRICES_TRAIN = {
    "lightgbm_base_46": X_base_train,
    "lightgbm_exp5b": X_exp5b_train,
    "lightgbm_exp5b_llm": X_llm_train,
    "lightgbm_exp5b_bert": X_bert_train,
}
MATRICES_TEST_SHAPE = {
    "lightgbm_base_46": X_base_test,
    "lightgbm_exp5b": X_exp5b_test,
    "lightgbm_exp5b_llm": X_llm_test,
    "lightgbm_exp5b_bert": X_bert_test,
}
winner = str(cv_results.iloc[0]["modelo"])
if winner not in MATRICES_TRAIN:
    raise RuntimeError(f"Ganador no exportable con este protocolo: {winner}")
X_final_train, X_final_test = MATRICES_TRAIN[winner], MATRICES_TEST_SHAPE[winner]
proba_oof = oof_probas[winner]
assert len(X_final_train) == len(y_train) and len(X_final_test) == len(X_test)
assert list(X_final_train.columns) == list(X_final_test.columns)


def aplicar_politica(
    proba: np.ndarray,
    pesos: np.ndarray | None = None,
    threshold_a1: float = THRESHOLD_A1,
) -> np.ndarray:
    assert threshold_a1 >= 0.20
    if pesos is None:
        pesos = np.ones(5)
    pred = np.argmax(proba * np.asarray(pesos).reshape(1, -1), axis=1) + 1
    pred[proba[:, 0] >= threshold_a1] = 1
    return pred.astype(int)


def objetivo(pesos: np.ndarray) -> float:
    return -macro_f1(y_train, aplicar_politica(proba_oof, np.clip(pesos, 0.05, 20.0)))


pred_argmax = np.argmax(proba_oof, axis=1) + 1
pred_a1 = aplicar_politica(proba_oof)
res = minimize(
    objetivo,
    x0=np.ones(5),
    method="Nelder-Mead",
    options={"maxiter": 10000, "xatol": 1e-6, "fatol": 1e-6, "adaptive": True},
)
pesos_finales = np.clip(res.x, 0.05, 20.0)
pred_final_oof = aplicar_politica(proba_oof, pesos_finales)
calibracion = pd.DataFrame(
    [
        {"politica": "argmax", **metricas(y_train, pred_argmax)},
        {"politica": "threshold_a1_0.20", **metricas(y_train, pred_a1)},
        {
            "politica": "threshold_a1_0.20_pesos_oof",
            **metricas(y_train, pred_final_oof),
        },
    ]
)
assert (
    calibracion.loc[
        calibracion["politica"] == "threshold_a1_0.20_pesos_oof", "recall_a1"
    ].iloc[0]
    >= calibracion.loc[calibracion["politica"] == "argmax", "recall_a1"].iloc[0]
)
print("Ganador:", winner)
print("Pesos finales:", [round(float(x), 4) for x in pesos_finales])
calibracion

Ganador: lightgbm_exp5b_bert
Pesos finales: [1.0589, 1.0807, 1.0749, 0.8892, 0.997]


,politica,macro_f1,recall_a1,precision_a1,recall_a2,precision_a2
0,argmax,0.575340,0.705126,0.675052,0.659084,0.670486
1,threshold_a1_0.20,0.553277,0.816220,0.488718,0.597850,0.677035
2,threshold_a1_0.20_pesos_oof,0.555034,0.816220,0.488718,0.599585,0.676274


## 9. Entrenamiento final y exportacion

Con el modelo, columnas y politica ya congelados, se entrena LightGBM con todo train. Se guardan los artefactos necesarios para el notebook 08 y para la produccion academica.

In [10]:
best_iter = (
    cv_results.loc[cv_results["modelo"] == winner, "best_iteration_mean"]
    .fillna(LGBM_PARAMS.get("n_estimators", 1000))
    .iloc[0]
)
final_params = {k: v for k, v in LGBM_PARAMS.items() if k != "n_estimators"}
final_params.update(
    {
        "n_estimators": max(int(round(float(best_iter))), 50),
        "objective": "multiclass",
        "num_class": 5,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "verbose": -1,
    }
)
final_model = LGBMClassifier(**final_params)
final_model.fit(X_final_train, y_train - 1, sample_weight=sample_weights_train)

proba_check = final_model.predict_proba(X_final_train.iloc[:10])
assert proba_check.shape == (10, 5)
assert np.allclose(proba_check.sum(axis=1), 1.0, atol=1e-5)
assert final_model.n_features_in_ == X_final_train.shape[1]
assert set(aplicar_politica(proba_check, pesos_finales)).issubset({1, 2, 3, 4, 5})


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


cv_results.to_parquet(OUT["cv_results"], index=False)
oof_export = pd.concat(oof_tables, ignore_index=True)
oof_export["row_id"] = oof_export.groupby("modelo").cumcount()
oof_export.to_parquet(OUT["oof_predictions"], index=False)
joblib.dump(final_model, OUT["model"], compress=3)
OUT["feature_list"].write_text(
    json.dumps(
        {
            "model_name": winner,
            "features": list(X_final_train.columns),
            "n_features": X_final_train.shape[1],
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)
OUT["thresholds"].write_text(
    json.dumps(
        {
            "policy": "threshold_a1_then_weighted_argmax",
            "threshold_a1": THRESHOLD_A1,
            "class_weights": [float(x) for x in pesos_finales],
            "calibrated_on": "OOF train probabilities only",
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)
metadata = {
    "created_at_unix": int(time.time()),
    "random_state": RANDOM_STATE,
    "cv": "StratifiedGroupKFold grouped by patient",
    "winner": winner,
    "n_train": len(y_train),
    "n_test_loaded_for_shape_only": len(X_test),
    "test_usage": "X_test solo para shapes/columnas; y_test no se carga",
    "lgbm_params": final_params,
    "bert_svd_path": str(bert_svd_path.relative_to(PROJECT_ROOT))
    if bert_svd_path
    else None,
    "outputs": {k: str(v.relative_to(PROJECT_ROOT)) for k, v in OUT.items()},
}
metadata["hashes"] = {
    k: sha256_file(v) for k, v in OUT.items() if v.exists() and k != "metadata"
}
OUT["metadata"].write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8"
)

model_v = joblib.load(OUT["model"])
features_v = json.loads(OUT["feature_list"].read_text(encoding="utf-8"))["features"]
thresholds_v = json.loads(OUT["thresholds"].read_text(encoding="utf-8"))
assert model_v.n_features_in_ == len(features_v)
assert list(X_final_train.columns) == features_v
assert len(thresholds_v["class_weights"]) == 5 and thresholds_v["threshold_a1"] >= 0.20
assert model_v.predict_proba(X_final_train[features_v].iloc[:1]).shape == (1, 5)
for name, path in OUT.items():
    print(f"{name:<16} -> {path.relative_to(PROJECT_ROOT)}")

model            -> models\lgbm_bert_final.joblib
thresholds       -> models\thresholds.json
feature_list     -> models\feature_list.json
cv_results       -> data\processed\cv_results.parquet
oof_predictions  -> data\processed\oof_predictions.parquet
metadata         -> models\model_training_metadata.json


## 10. Cierre

El resultado de este notebook es una configuracion congelada: modelo, columnas, probabilidades OOF, resultados CV, thresholds y metadata. El siguiente notebook solo debe cargar estos artefactos y evaluar una vez en test temporal.